In [4]:
list1 = ['a', 'b', 'c', 'd']
list2 = ['d', 'f']

In [1]:
import pandas as pd


In [5]:
a = pd.Series([1, 2, 3, 4])

In [8]:
pd.DataFrame(a)[[0]]

,0
0,1
1,2
2,3
3,4


In [9]:
dictionary = {
    'a': 1,
    'b': 2,
    'c': 3,
    'd': 4
}

In [19]:
categorical_columns = ['a']
numerical_columns = ['c']
        
all_columns = set(['a', 'b', 'c'])
no_type_columns = all_columns - (set(categorical_columns) | set(numerical_columns))

print(no_type_columns)

{'b'}


In [15]:
sorted_items = sorted(dictionary.items(), key=lambda x: x[1], reverse=True)
    
# 步骤2：截取前n个元素（防止n超过字典长度）
top_n_items = [item[0] for item in sorted_items[:3]]

In [16]:
top_n_items

['d', 'c', 'b']

In [5]:
[x for x in list1 if x not in list2]

['a', 'b', 'c']

In [11]:
max({'a':1, 'b':10}.values())

10

In [13]:
not []

True

In [14]:
a = {}

In [24]:
b = a.get('1', 1)

In [26]:
type(b)

int

In [1]:
from module_transform import Transform


In [2]:
t = Transform()

In [3]:
import pandas as pd

In [4]:
t._transform_numerical_attribute(
    pd.DataFrame([1, 2, 3], columns=['A']),
    change={
        'beta_O': 0,
        'beta_Y': 0
    }
)

1


,A
0,0.0
1,0.269231
2,1.0


In [8]:
import pandas as pd
import numpy as np
from itertools import combinations

# 模拟数据准备
np.random.seed(42)
transformed_df = pd.DataFrame({
    'gender': np.random.choice([0, 1], size=100),
    'age_group': np.random.choice([1, 2, 3], size=100)
})

O = pd.DataFrame({
    'target': np.random.randint(0,2, size=100)
})

# 参数设置
selected_attribute = 'age_group'
selected_label_O = 'target'
PARAMS_MAIN_BM_REBIN_METHOD = 'r1'  # 可切换为'r2'测试不同方法
ZORDER=1
changed_dict = {}

# 核心逻辑
df_data_bm_temp = pd.concat([
    transformed_df.reset_index(drop=True),
    O.reset_index(drop=True)
], axis=1)

attr_bias = selected_attribute
label_O = selected_label_O

def compute_r1_rebin(df, attr_bias, label_O, zorder=1):
    diff_change = []
    
    for i, j in combinations(df[label_O].unique(), 2):
        df_0 = df[df[label_O] == i].groupby(attr_bias).size()
        df_1 = df[df[label_O] == j].groupby(attr_bias).size()
        
        df_0_sum = max(1, len(df_0))
        df_1_sum = max(1, len(df_1))
        
        prop_0 = df_0 / df_0_sum
        prop_1 = df_1 / df_1_sum
        
        diff_matrix = prop_1 - prop_0
        
        group_diffs = []
        for group in diff_matrix.index:
            other_groups = [g for g in diff_matrix.index if g != group]
            for other in other_groups:
                diff_val = diff_matrix.loc[group] - diff_matrix.loc[other]
                group_diffs.append({
                    'group1': group,
                    'group2': other,
                    'diff': diff_val
                })
        
        df_diffs = pd.DataFrame(group_diffs)
        
        df_diffs_sorted = df_diffs.sort_values(by='diff', ascending=False)
        print(df_diffs_sorted)
        if zorder <= len(df_diffs_sorted):
            selected = df_diffs_sorted.iloc[zorder-1]
            change_temp = {selected['group1']: selected['group2']}
        else:
            selected = df_diffs_sorted.iloc[0]
            change_temp = {selected['group1']: selected['group2']}
        
        diff_change.append({
            'change': change_temp,
            'diff': selected['diff']
        })
    
    if not diff_change:
        return {}
    
    df_results = pd.DataFrame(diff_change)
    df_results_sorted = df_results.sort_values(by='diff', ascending=False)
    
    if zorder <= len(df_results_sorted):
        return df_results_sorted.iloc[zorder-1]['change']
    else:
        return df_results_sorted.iloc[0]['change']

if PARAMS_MAIN_BM_REBIN_METHOD == 'r1':
    change = compute_r1_rebin(
            df_data_bm_temp,
            attr_bias,
            label_O,
            zorder=ZORDER
        )
    
elif PARAMS_MAIN_BM_REBIN_METHOD == 'r2':
    df_c = df_data_bm_temp[attr_bias].value_counts().sort_values()
    change = {df_c.index[1]: df_c.index[0]}

# 更新changed_dict
if attr_bias not in changed_dict:
    changed_dict[attr_bias] = {}
    
converted_change = {int(k): int(v) for k, v in change.items()}
changed_dict[attr_bias].update(converted_change)

# 打印结果验证
print("changed_dict:", changed_dict)
print("\n验证数据：")
print("原始数据分布：")
print(df_data_bm_temp[attr_bias].value_counts())
print("\ntarget列分布：")
print(df_data_bm_temp[label_O].value_counts())

   group1  group2      diff
5       3       2  2.333333
0       1       2  1.666667
4       3       1  0.666667
1       1       3 -0.666667
2       2       1 -1.666667
3       2       3 -2.333333
changed_dict: {'age_group': {3: 2}}

验证数据：
原始数据分布：
age_group
3    39
1    35
2    26
Name: count, dtype: int64

target列分布：
target
1    53
0    47
Name: count, dtype: int64
